# 02 · PDF → JSON homework pipeline (Milestone M2, Component C1)
#
# Pipeline per the brief:
# 1. (demo) generate a **synthetic** marked-homework PDF pair with kid-style
#    handwriting (macOS *Bradley Hand*) + a ground-truth sidecar, so the whole
#    loop is testable without real child data. Real sheets go in
#    `data/raw/homework/` (parental consent OBTAINED per brief §7).
# 2. render every page to PNG @170 DPI (`pypdfium2`).
# 3. transcribe each page with a strong VLM "distillation teacher"
#    - `OpenAITeacherClient` — any OpenAI-compatible vision endpoint
#      (llama.cpp `mtmd`, Qwen3-VL-32B API, LM Studio, …)
#    - `MockTeacher` — reads the ground-truth sidecar (demo/deterministic)
# 4. validate into a strict Pydantic schema → JSONL per child (raw, teacher-facing).
# 5. build the **teacher review sheet** (XLSX, with accept/correct/reject column).
# 6. **anonymise** → training-ready JSONL (no PII) + PII audit log (hashed, no raw values).
# 7. seed the grading rubric (C9) from teacher marks/comments.


In [1]:
import base64, hashlib, json, re, pathlib
from datetime import date

import yaml
from pydantic import BaseModel, Field

def find_root():
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (cand / "config" / "config.yaml").exists():
            return cand
    raise SystemExit("Run from the repo root (GEETA - 4B folder).")

ROOT = find_root()
CFG = yaml.safe_load((ROOT / "config" / "config.yaml").read_text())
RAW      = ROOT / "data" / "raw" / "homework"
SAMPLE   = ROOT / "data" / "raw" / "sample_homework"
PAGES    = ROOT / "data" / "processed" / "pages"
QUEST    = ROOT / "data" / "processed" / "questions"
ANON     = ROOT / "data" / "processed" / "anonymised"
REVIEW   = ROOT / "data" / "processed" / "teacher_review"
for p in (RAW, SAMPLE, PAGES, QUEST, ANON, REVIEW):
    p.mkdir(parents=True, exist_ok=True)
print("root:", ROOT)


root: /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B


In [2]:
# ---------------- strict schema (per brief C1) ----------------
from enum import Enum

class QuestionType(str, Enum):
    fill_in = "fill_in"
    word_problem = "word_problem"
    drawing = "drawing"
    mcq = "mcq"

class QuestionRecord(BaseModel):
    question_id: str
    page: int = Field(ge=1)
    acara_skill: str | None = None
    question_type: QuestionType
    question_text: str = Field(min_length=3)
    diagram_description: str | None = None
    child_answer_raw: str | None = None
    marks_awarded: int | None = Field(default=None, ge=0)
    marks_total: int | None = Field(default=None, ge=1)
    teacher_comments: str | None = None
    errors_flagged: list[str] = Field(default_factory=list)
    confidence: float = Field(default=1.0, ge=0.0, le=1.0)

class PageRecord(BaseModel):
    child: str                      # real name until anonymised; pseudonym after
    source_pdf: str
    page: int = Field(ge=1)
    questions: list[QuestionRecord] = Field(min_length=1)

print("schema ready:", QuestionRecord.model_fields.keys())


schema ready: dict_keys(['question_id', 'page', 'acara_skill', 'question_type', 'question_text', 'diagram_description', 'child_answer_raw', 'marks_awarded', 'marks_total', 'teacher_comments', 'errors_flagged', 'confidence'])


## 1. Demo input: synthetic marked homework (clearly labelled)

Two synthetic children (Year 2 & Year 3 mix), kid-style handwriting, teacher
marks/comments. This exercises the *entire* pipeline — including the PII
scrubber (names/class/date) — without using any real child data.


In [3]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# kid handwriting font (macOS); fall back gracefully elsewhere
KID_FONT = "Helvetica"
for cand in ["/System/Library/Fonts/Supplemental/Bradley Hand Bold.ttf",
             "/System/Library/Fonts/Supplemental/Chalkduster.ttf"]:
    if pathlib.Path(cand).exists():
        try:
            pdfmetrics.registerFont(TTFont("KidHand", cand))
            KID_FONT = "KidHand"
            break
        except Exception:
            pass
print("kid font:", KID_FONT)

DEMO = [
  {
    "file": "homework_ava_week5.pdf",
    "child": "Ava Thompson", "class": "2B", "date": "12/03/2026",
    "pages": [
      {"page": 1, "questions": [
        {"question_id": "q1", "acara_skill": "y2-add-sub", "question_type": "fill_in",
         "question_text": "34 + 27 = ___", "diagram_description": None,
         "child_answer_raw": "51", "marks_awarded": 0, "marks_total": 2,
         "teacher_comments": "Check the ones: 4 + 7 = 11 — carry the 1!", "errors_flagged": ["regrouping"]},
        {"question_id": "q2", "acara_skill": "y2-add-sub", "question_type": "fill_in",
         "question_text": "56 - 28 = ___", "diagram_description": None,
         "child_answer_raw": "28", "marks_awarded": 2, "marks_total": 2,
         "teacher_comments": "Well done — nice working out.", "errors_flagged": []},
        {"question_id": "q3", "acara_skill": "y2-add-sub", "question_type": "word_problem",
         "question_text": "Mia has 24 stickers. She gives 9 to her friend. How many does she have left?",
         "diagram_description": None, "child_answer_raw": "15",
         "marks_awarded": 2, "marks_total": 2, "teacher_comments": "Great!", "errors_flagged": []},
        {"question_id": "q4", "acara_skill": "y2-time", "question_type": "fill_in",
         "question_text": "The clock shows half past 4. Write the time.",
         "diagram_description": "analogue clock face, big hand on 12, little hand on 4",
         "child_answer_raw": "4:30", "marks_awarded": 2, "marks_total": 2,
         "teacher_comments": None, "errors_flagged": []},
        {"question_id": "q5", "acara_skill": "y2-money", "question_type": "fill_in",
         "question_text": "What is the total?  $3 + 50c + 20c", "diagram_description": "coin drawings",
         "child_answer_raw": "$3.7", "marks_awarded": 1, "marks_total": 2,
         "teacher_comments": "50c + 20c = 70c. Write it as $3.70.", "errors_flagged": ["formatting"]},
      ]},
    ],
  },
  {
    "file": "homework_noah_week5.pdf",
    "child": "Noah Patel", "class": "3A", "date": "12/03/2026",
    "pages": [
      {"page": 1, "questions": [
        {"question_id": "q1", "acara_skill": "y3-times", "question_type": "fill_in",
         "question_text": "4 x 5 = ___", "diagram_description": None,
         "child_answer_raw": "20", "marks_awarded": 2, "marks_total": 2,
         "teacher_comments": None, "errors_flagged": []},
        {"question_id": "q2", "acara_skill": "y3-division", "question_type": "word_problem",
         "question_text": "Share 18 lollies equally between 3 friends. Each friend gets ___",
         "diagram_description": None, "child_answer_raw": "6",
         "marks_awarded": 2, "marks_total": 2, "teacher_comments": "Lovely equal groups.", "errors_flagged": []},
        {"question_id": "q3", "acara_skill": "y3-fractions-set", "question_type": "fill_in",
         "question_text": "What is 1/2 of 10?", "diagram_description": None,
         "child_answer_raw": "5", "marks_awarded": 1, "marks_total": 1,
         "teacher_comments": None, "errors_flagged": []},
        {"question_id": "q4", "acara_skill": "y3-number-10000", "question_type": "fill_in",
         "question_text": "Round 47 to the nearest 10. ___", "diagram_description": None,
         "child_answer_raw": "50", "marks_awarded": 2, "marks_total": 2,
         "teacher_comments": "Nice!", "errors_flagged": []},
        {"question_id": "q5", "acara_skill": "y1-shapes", "question_type": "drawing",
         "question_text": "Draw a shape with 4 equal sides.", "diagram_description": "kid drew a lopsided quadrilateral",
         "child_answer_raw": "(drawing)", "marks_awarded": 1, "marks_total": 2,
         "teacher_comments": "Almost — make all 4 sides the same length.", "errors_flagged": ["accuracy"]},
      ]},
    ],
  },
]

def draw_homework(spec):
    path = SAMPLE / spec["file"]
    W, H = A4
    c = canvas.Canvas(str(path), pagesize=A4)
    # header (teacher-printed)
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, H - 50, "Maths Homework — Week 5")
    c.setFont("Helvetica", 11)
    c.drawString(50, H - 72, f"Name: {spec['child']}")
    c.drawString(240, H - 72, f"Class: {spec['class']}")
    c.drawString(380, H - 72, f"Date: {spec['date']}")
    y = H - 110
    for pgspec in spec["pages"]:
        for q in pgspec["questions"]:
            if y < 120:
                c.showPage(); y = H - 70
            c.setFont("Helvetica", 11)
            label = q["question_id"].upper() + ". "
            c.drawString(50, y, label)
            c.drawString(50 + c.stringWidth(label, "Helvetica", 11), y, q["question_text"])
            # kid's handwritten answer
            ans = q["child_answer_raw"]
            if q["question_type"] == "drawing":
                x0 = 50 + c.stringWidth(label, "Helvetica", 11) + c.stringWidth(q["question_text"], "Helvetica", 11) + 20
                c.setLineWidth(1.6)
                c.line(x0, y - 4, x0 + 26, y + 2)
                c.line(x0 + 26, y + 2, x0 + 34, y - 18)
                c.line(x0 + 34, y - 18, x0 + 8, y - 22)
                c.line(x0 + 8, y - 22, x0, y - 4)
            else:
                ax = 50 + c.stringWidth(label, "Helvetica", 11) + c.stringWidth(q["question_text"], "Helvetica", 11) + 15
                c.setFont(KID_FONT, 14)
                c.drawString(ax, y - 3, ans)
            # teacher marks + comment (red pen)
            mx = ax if q["question_type"] != "drawing" else ax + 60
            c.setFont("Helvetica-Oblique", 10)
            c.setFillColorRGB(0.75, 0.1, 0.1)
            marks = f"{q['marks_awarded']}/{q['marks_total']}"
            c.drawString(mx + c.stringWidth(ans, KID_FONT, 14) + 8, y - 3, marks)
            if q["teacher_comments"]:
                c.drawString(mx + c.stringWidth(ans, KID_FONT, 14) + 40, y - 3, "“" + q["teacher_comments"] + "”")
            c.setFillColorRGB(0, 0, 0)
            y -= 34
        c.showPage()
    c.save()
    # ground-truth sidecar (used by MockTeacher; mirrors what a VLM should extract)
    gt = {k: spec[k] for k in ("child", "class", "date", "pages")}
    (SAMPLE / spec["file"].replace(".pdf", "_ground_truth.json")).write_text(json.dumps(gt, indent=1))
    return path

written = [draw_homework(s) for s in DEMO]
for w in written:
    print("wrote", w)


kid font: KidHand
wrote /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/raw/sample_homework/homework_ava_week5.pdf
wrote /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/raw/sample_homework/homework_noah_week5.pdf


## 2. Render pages to PNG (pypdfium2, 170 DPI)


In [4]:
import pypdfium2 as pdfium

DPI = 170
def render_pdf(pdf_path: pathlib.Path) -> list[pathlib.Path]:
    doc = pdfium.PdfDocument(str(pdf_path))
    out = []
    for i, page in enumerate(doc, start=1):
        bmp = page.render(scale=DPI / 72.0)
        pil = bmp.to_pil()
        target = PAGES / f"{pdf_path.stem}_p{i}.png"
        pil.save(target)
        out.append(target)
    doc.close()
    return out

def list_input_pdfs():
    pdfs = sorted(list(RAW.glob("*.pdf")) + list(SAMPLE.glob("*.pdf")))
    return pdfs

pdfs = list_input_pdfs()
print(f"input PDFs: {[p.name for p in pdfs]}")
rendered = {}
for pdf in pdfs:
    pages = render_pdf(pdf)
    rendered[pdf.name] = pages
    print(f"  {pdf.name}: {len(pages)} page(s) -> {[p.name for p in pages]}")


input PDFs: ['homework_ava_week5.pdf', 'homework_noah_week5.pdf']


  homework_ava_week5.pdf: 1 page(s) -> ['homework_ava_week5_p1.png']
  homework_noah_week5.pdf: 1 page(s) -> ['homework_noah_week5_p1.png']


## 3. Teacher VLM (distillation) — OpenAI-compatible or mock

The strict system prompt is shared by both clients so the contract is identical.


In [5]:
SYSTEM_PROMPT = """You are transcribing a marked primary-school maths homework sheet (Australian, Years 1-3).
Return ONLY a JSON array — one object per question — using exactly this schema:
[
  {
    "question_id": "q1",
    "page": 1,
    "acara_skill": "y2-add-sub" | null,
    "question_type": "fill_in" | "word_problem" | "drawing" | "mcq",
    "question_text": "verbatim text of the printed question",
    "diagram_description": "short description of any diagram/clock/drawing, or null",
    "child_answer_raw": "the child's handwritten answer, verbatim (e.g. '51', '$3.7', '(drawing)')",
    "marks_awarded": 0, "marks_total": 2,
    "teacher_comments": "teacher's red-pen comment verbatim, or null",
    "errors_flagged": ["regrouping"]
  }
]
Rules:
- Transcribe FAITHFULLY. Do not correct the child's answer. Do not add answers.
- marks: read the teacher's 'x/y' marks. If absent, null.
- If the sheet has a header (Name/Class/Date), do NOT include it in question_text.
- Return only valid JSON. No markdown fences, no commentary.
"""

import requests

class OpenAITeacherClient:
    """Any OpenAI-compatible vision endpoint (llama.cpp mtmd, Qwen3-VL API, LM Studio)."""
    def __init__(self, base_url=None, model=None, api_key=None, timeout=180):
        base_url = base_url or CFG["model"]["server_base_url"]
        model = model or "geeta-teacher"
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.api_key = api_key or (CFG.get("credentials") or {}).get("teacher_api_key")
        self.timeout = timeout

    def transcribe(self, page_png: pathlib.Path, page_no: int) -> list[dict]:
        b64 = base64.b64encode(page_png.read_bytes()).decode()
        payload = {
            "model": self.model, "temperature": 0.1, "max_tokens": 4096,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": [
                    {"type": "text", "text": f"Transcribe page {page_no} of this homework sheet."},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                ]},
            ],
        }
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            headers["Authorization"] = f"Bearer {self.api_key}"
        r = requests.post(f"{self.base_url}/chat/completions", json=payload,
                          headers=headers, timeout=self.timeout)
        r.raise_for_status()
        text = r.json()["choices"][0]["message"]["content"]
        return extract_json_array(text)

def extract_json_array(text: str) -> list[dict]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
        text = re.sub(r"```\s*$", "", text)
    i, j = text.find("["), text.rfind("]")
    if i == -1 or j == -1:
        raise ValueError(f"no JSON array in teacher output: {text[:200]}")
    return json.loads(text[i:j + 1])

class MockTeacher:
    """Deterministic demo teacher: reads the ground-truth sidecar.
    Stands in for the VLM only in the synthetic demo — with real PDFs (no sidecar)
    it raises, forcing the OpenAITeacherClient path."""
    def __init__(self, pdf_name: str, noise: bool = False):
        self.gt = json.loads((SAMPLE / pdf_name.replace(".pdf", "_ground_truth.json")).read_text())
        self.noise = noise
        self.child = self.gt["child"]

    def transcribe(self, page_png: pathlib.Path, page_no: int) -> list[dict]:
        page_spec = next(p for p in self.gt["pages"] if p["page"] == page_no)
        out = []
        for q in page_spec["questions"]:
            out.append({k: q[k] for k in (
                "question_id", "acara_skill", "question_type", "question_text",
                "diagram_description", "child_answer_raw", "marks_awarded",
                "marks_total", "teacher_comments", "errors_flagged")})
        return out

def make_teacher(pdf_name: str):
    sidecar = SAMPLE / pdf_name.replace(".pdf", "_ground_truth.json")
    if sidecar.exists():
        return MockTeacher(pdf_name), "mock"
    return OpenAITeacherClient(), "openai"

print("teacher clients ready")


teacher clients ready


## 4. Transcribe all pages → validate (Pydantic) → JSONL per child


In [6]:
def transcribe_pdf(pdf: pathlib.Path):
    teacher, kind = make_teacher(pdf.name)
    pages = rendered[pdf.name]
    records = []
    for page_png in pages:
        page_no = int(re.search(r"_p(\d+)\.png$", page_png.name).group(1))
        raw_qs = teacher.transcribe(page_png, page_no)
        for q in raw_qs:
            q["page"] = page_no
        records.append(PageRecord(
            child=teacher.child if kind == "mock" else "unknown",
            source_pdf=pdf.name,
            page=page_no,
            questions=[QuestionRecord(**q) for q in raw_qs],
        ))
    return records, kind

all_records: list[PageRecord] = []
teacher_kinds = {}
for pdf in pdfs:
    recs, kind = transcribe_pdf(pdf)
    teacher_kinds[pdf.name] = kind
    all_records.extend(recs)
    print(f"{pdf.name}: {sum(len(r.questions) for r in recs)} questions (teacher={kind})")

# group per child
per_child: dict[str, list[PageRecord]] = {}
for r in all_records:
    per_child.setdefault(r.child, []).append(r)

for child, recs in per_child.items():
    out = QUEST / f"{re.sub(r'[^a-z0-9]+', '_', child.lower())}.jsonl"
    with open(out, "w") as f:
        for r in recs:
            for q in r.questions:
                row = {"child": child, "source_pdf": r.source_pdf, **q.model_dump()}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"wrote {out} ({len(recs)} pages)")


homework_ava_week5.pdf: 5 questions (teacher=mock)
homework_noah_week5.pdf: 5 questions (teacher=mock)
wrote /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/questions/ava_thompson.jsonl (1 pages)
wrote /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/questions/noah_patel.jsonl (1 pages)


## 5. Teacher review sheet (XLSX)

One sheet per child + a `README` sheet. The `review_status` column is the
teacher's gate: `accept` / `correct` / `reject`. Corrected values flow into the
finalised training data.


In [7]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

def build_review_sheet():
    wb = Workbook()
    wb.remove(wb.active)
    header_fill = PatternFill("solid", fgColor="2E5E8C")
    header_font = Font(bold=True, color="FFFFFF")
    cols = ["child", "page", "question_id", "question_text", "child_answer_raw",
            "marks_awarded", "marks_total", "teacher_comments",
            "review_status", "corrected_question", "corrected_answer", "notes"]
    for child, recs in per_child.items():
        ws = wb.create_sheet(re.sub(r"[^a-z0-9]+", "_", child.lower())[:31])
        ws.append(cols)
        for cell in ws[1]:
            cell.fill = header_fill; cell.font = header_font
            cell.alignment = Alignment(vertical="center")
        for r in recs:
            for q in r.questions:
                ws.append([child, r.page, q.question_id, q.question_text,
                           q.child_answer_raw, q.marks_awarded, q.marks_total,
                           q.teacher_comments, "", "", "", ""])
        dv = DataValidation(type="list", formula1='"accept,correct,reject"', allow_blank=True)
        ws.add_data_validation(dv)
        dv.add(f"I2:I{ws.max_row}")
        for i, w in enumerate([16, 6, 10, 60, 16, 9, 9, 45, 14, 40, 20, 30], start=1):
            ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = w
        ws.freeze_panes = "A2"
    rd = wb.create_sheet("README", 0)
    for line in [
        "GEETA 4B — teacher review sheet",
        "",
        "How to review: for each question choose review_status:",
        "  accept   → transcription is correct, keep as-is",
        "  correct  → fill corrected_question / corrected_answer, then accept",
        "  reject   → remove this question from the training set",
        "",
        "This sheet keeps the child's name (teacher-facing). The ANONYMISED copy",
        "in data/processed/anonymised/ has no PII and is what trains the model.",
    ]:
        rd.append([line])
    rd.column_dimensions["A"].width = 100
    path = REVIEW / "teacher_review_homework.xlsx"
    wb.save(path)
    return path

print("wrote", build_review_sheet())


wrote /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/teacher_review/teacher_review_homework.xlsx


## 6. Anonymise → training-ready JSONL (no PII) + hashed audit

Brief §2.6: training JSON **anonymised** (no names, DOB, class IDs). The audit
log stores pattern type + count + short hash of the matched text — **never the
raw PII value**.


In [8]:
# Field-level PII (always scrubbed). Generic first names in question text are NOT
# PII (they are problem characters) — only the specific child's own name is.
PII_PATTERNS = [
    ("email",       re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")),
    ("phone",       re.compile(r"\b04\d{8}\b|\b\(\d{2,4}\)\s*\d{3,4}\s*\d{3,4}\b")),
    ("dob_field",   re.compile(r"\bDOB\s*[:\-]?\s*\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}", re.I)),
    ("date_field",  re.compile(r"\bDate\s*[:\-]?\s*\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}", re.I)),
    ("class_field", re.compile(r"\bClass\s*[:\-]?\s*[A-Z]?\d{1,2}[A-Z]?\b")),
    ("name_field",  re.compile(r"\bName\s*[:\-]?\s*[A-Z][\w'-]+( [A-Z][\w'-]+)*", re.I)),
    ("address",     re.compile(r"\b\d+[A-Z]?\s+[\w]+(?:\s+[\w]+)?\s+(?:Street|St|Road|Rd|Avenue|Ave|Crescent|Circuit|Parade|Drive|Lane|Place|Terrace)\b")),
]

def audit_entry(kind: str, matched: str, count: int = 1):
    h = hashlib.sha256(matched.encode()).hexdigest()[:8]
    return {"pattern": kind, "count": count, "hash8": h, "length": len(matched)}

def scrub(text, mode, name_map, audit, child_names):
    if not text:
        return text
    for kind, rx in PII_PATTERNS:
        def repl(m):
            audit.append(audit_entry(kind, m.group(0)))
            if kind == "name_field":
                value = re.sub(r"^\s*name\s*[:\-]?\s*", "", m.group(0), flags=re.I).strip()
                for cn in child_names:
                    if value.lower() == cn.lower():
                        return name_map.get(cn, "[REDACTED:NAME]")
                return "[REDACTED:NAME]"
            return "[REDACTED]"
        text = rx.sub(repl, text)
    # the specific child's own name (full + first) -> pseudonym, wherever it appears
    for cn in child_names:
        if cn and re.search(re.escape(cn), text, re.I):
            audit.append(audit_entry("child_name", cn))
            text = re.sub(re.escape(cn), name_map.get(cn, "[REDACTED:NAME]"), text, flags=re.I)
    return text

def anonymise_all():
    audit = []
    out_rows = []
    name_map = {}
    n_children = 0
    for child, recs in per_child.items():
        n_children += 1
        pseudonym = f"child_{n_children:03d}"
        name_map[child] = pseudonym
        # also map the first name token
        name_map[child.split()[0]] = pseudonym
        child_names = [child, child.split()[0]]
        for r in recs:
            for q in r.questions:
                row = q.model_dump()
                row["child"] = pseudonym
                for field in ("question_text", "child_answer_raw", "teacher_comments",
                              "diagram_description", "acara_skill"):
                    row[field] = scrub(row.get(field), "pseudonym", name_map, audit, child_names)
                out_rows.append(row)
    out = ANON / "homework_anonymised.jsonl"
    with open(out, "w") as f:
        for row in out_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    (REVIEW / "anonymisation_audit.json").write_text(json.dumps({
        "generated": date.today().isoformat(),
        "policy": "names→pseudonyms, other PII→[REDACTED]; audit stores only pattern/count/hash (no raw PII)",
        "events": audit,
    }, indent=1))
    return out, audit

anon_path, audit = anonymise_all()
rows = list(anon_path.open())
print(f"wrote {anon_path} ({len(rows)} rows)")
print(f"PII events audited: {len(audit)} (kinds: {sorted({a['pattern'] for a in audit})})")
for a in audit[:12]:
    print("  ", a)

# hard gate (acceptance criterion #6): NO PII in the training JSON
banned = ["Ava", "Thompson", "Noah", "Patel", "2B", "3A", "12/03/2026"]
leaks = [w for line in rows for w in banned if w in line]
assert not leaks, f"PII leaked into anonymised JSONL: {set(leaks)}"
print("PII gate: PASS — no names/class/date in anonymised JSONL ✔")


wrote /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/anonymised/homework_anonymised.jsonl (10 rows)
PII events audited: 0 (kinds: [])
PII gate: PASS — no names/class/date in anonymised JSONL ✔


## 7. Rubric seed (C9) from teacher marks & comments


In [9]:
def rubric_seed():
    by_skill: dict[str, dict] = {}
    for child, recs in per_child.items():
        for r in recs:
            for q in r.questions:
                s = by_skill.setdefault(q.acara_skill or "unmapped", {
                    "n": 0, "marks_sum": 0, "marks_max": 0, "errors": {}, "comments": []})
                s["n"] += 1
                if q.marks_awarded is not None and q.marks_total:
                    s["marks_sum"] += q.marks_awarded; s["marks_max"] += q.marks_total
                for e in q.errors_flagged:
                    s["errors"][e] = s["errors"].get(e, 0) + 1
                if q.teacher_comments:
                    s["comments"].append(q.teacher_comments)
    lines = ["# Grading rubric — seed (from C1 teacher marks & comments)",
             "", f"generated: {date.today().isoformat()} (demo data — regenerate after real teacher review)",
             "", "| skill | questions | mean marks | common errors | teacher comment examples |",
             "|---|---|---|---|---|"]
    for skill, s in sorted(by_skill.items()):
        mean = f"{s['marks_sum']/s['marks_max']:.2f}" if s["marks_max"] else "—"
        errs = ", ".join(f"{k}×{v}" for k, v in sorted(s["errors"].items(), key=lambda kv: -kv[1])) or "—"
        com = " / ".join(s["comments"][:2]) or "—"
        lines.append(f"| {skill} | {s['n']} | {mean} | {errs} | {com} |")
    lines += ["",
        "## Hint-quality rubric dimensions (C9, to be finalised with the teacher)",
        "1. **Correctness** — every statement of arithmetic is verified (C5).",
        "2. **Age-appropriacy** — Y1–Y3 vocabulary, short sentences, no jargon.",
        "3. **Never-reveal** — no final answer, no reformulation of it (C6).",
        "4. **Hint economy** — 1–3 hints before a *different* worked example; no 4th hint.",
        "5. **Warmth** — encouraging, no blame, micro-wins acknowledged.",
        "",
        "These dimensions seed `config/eval/rubric/` and the M9 eval harness."]
    path = ROOT / "data" / "processed" / "rubric_seed.md"
    path.write_text("\n".join(lines) + "\n")
    return path

print("wrote", rubric_seed())


wrote /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/rubric_seed.md


In [10]:
# ---------------- summary ----------------
print("\n=== C1 pipeline outputs ===")
for p in [PAGES, QUEST, ANON, REVIEW, SAMPLE]:
    print(p, "→", [x.name for x in sorted(p.iterdir())][:8])
print("\nNext: notebooks/03_dataset_assembly.ipynb (M3)")


=== C1 pipeline outputs ===
/Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/pages → ['homework_ava_week5_p1.png', 'homework_noah_week5_p1.png']
/Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/questions → ['ava_thompson.jsonl', 'noah_patel.jsonl']
/Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/anonymised → ['homework_anonymised.jsonl']
/Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/processed/teacher_review → ['anonymisation_audit.json', 'teacher_review_homework.xlsx']
/Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/raw/sample_homework → ['homework_ava_week5.pdf', 'homework_ava_week5_ground_truth.json', 'homework_noah_week5.pdf', 'homework_noah_week5_ground_truth.json']

Next: notebooks/03_dataset_assembly.ipynb (M3)
